In [ ]:
pip install genieclust

In [ ]:
pip install pyclustering

In [ ]:
pip install openml

In [ ]:
pip install optuna

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
rng = np.random.default_rng()
from scipy.cluster import  hierarchy
from scipy.cluster.hierarchy import fcluster
import genieclust
from pyclustering.cluster.kmedoids import kmedoids
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_samples
import openml
from sklearn.model_selection import train_test_split
import time
import optuna
from sklearn.datasets import make_blobs, make_circles, make_moons, make_regression
from matplotlib.colors import ListedColormap
import cluster_master

## Dane rzeczywiste - wspomaganie w uczeniu nadzorowanym


Zbiór phoneme - klasyfikacja głosek na nosowe i zwykłe

In [ ]:
dataset = openml.datasets.get_dataset(1489)

X, y, _, _ = dataset.get_data(
    target=dataset.default_target_attribute
)
y = y.astype(int)-1

X_ptrain, X_test, y_ptrain, y_test = train_test_split(X, y, test_size=0.2, random_state=2137)
X_train, X_val, y_train, y_val = train_test_split(X_ptrain, y_ptrain, test_size=0.2, random_state=2137)

In [ ]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

In [ ]:
study_xgb_noclust = optuna.create_study(direction="maximize")
study_xgb_noclust.optimize(lambda trial: objective_xgb(trial), n_trials=100)

xgb_nocl = XGBClassifier(
    **study_xgb_noclust.best_params
)
xgb_nocl.fit(X_ptrain, y_ptrain)

In [ ]:
params = cluster_master.get_params_dict()
params['min_samples_lower'] = 200
params['min_samples_upper'] = 650
params['eps_upper'] = 1.8

cstudy = cluster_master.cluster_master_study(np.asarray(X_ptrain), "dbscan", "calinski-harabasz", 1000, params)

In [ ]:
clopt = DBSCAN(**cstudy.best_params)
clusters = clopt.fit_predict(X)


In [ ]:
Xc = X.assign(cluster=clusters)
X_ptrain, X_test, Xc_ptrain, Xc_test, y_ptrain, y_test = train_test_split(X, Xc, y, test_size=0.2, random_state=2137)
X_train, X_val, Xc_train, Xc_val, y_train, y_val = train_test_split(X_ptrain, Xc_ptrain, y_ptrain, test_size=0.2, random_state=2137)

In [ ]:
def objective_xgb(trial, clust = None):
  if clust is None:
    X = X_train
    y = y_train
    Xv = X_val
    yv = y_val
  else:
    X = Xc_train
    y = y_train
    Xv = Xc_val
    yv = y_val
  xgb_lr = trial.suggest_float(name="learning_rate", low=0, high=1)
  xgb_nest = trial.suggest_int("n_estimators", 10, 1000)
  xgb_subsample = trial.suggest_float(name="subsample", low=0, high=1)
  xgb_l1 = trial.suggest_float(name="reg_alpha", low=0, high=10)
  xgb_l2 = trial.suggest_float(name="reg_lambda", low=0, high=40)
  xgb_depth = trial.suggest_int("max_depth", 1, 20)
  xgb_gamma = trial.suggest_float(name="gamma", low=0, high=1)
  params = {
    "learning_rate": xgb_lr,
    "n_estimators": xgb_nest,
    "subsample": xgb_subsample,
    "max_depth": xgb_depth,
    "gamma": xgb_gamma,
    "reg_alpha": xgb_l1,
    "reg_lambda": xgb_l2,
    "eval_metric": "auc"
  }
  xgb = XGBClassifier(
      **params
  )

  xgb.fit(X, y)
  score = balanced_accuracy_score(xgb.predict(Xv), yv)
  return score

In [ ]:
study_xgb_noclust = optuna.create_study(direction="maximize")
study_xgb_noclust.optimize(lambda trial: objective_xgb(trial), n_trials=200)

xgb_nocl = XGBClassifier(
    **study_xgb_noclust.best_params
)
xgb_nocl.fit(X_ptrain, y_ptrain)

study_xgb_clust = optuna.create_study(direction="maximize")
study_xgb_clust.optimize(lambda trial: objective_xgb(trial, clust = "yes please"), n_trials=200)

xgb_clust = XGBClassifier(
    **study_xgb_clust.best_params
)
xgb_clust.fit(Xc_ptrain, y_ptrain)

In [ ]:
print(f"Balanced accuracy without clustering (XGB): {balanced_accuracy_score(xgb_nocl.predict(X_test), y_test)}")
print(f"Balanced accuracy with clustering (XGB): {balanced_accuracy_score(xgb_clust.predict(Xc_test), y_test)}")

In [ ]:
lr_noclust = LogisticRegression()
lr_noclust.fit(X_ptrain, y_ptrain)

lr_clust = LogisticRegression()
lr_clust.fit(Xc_ptrain, y_ptrain)

print(f"Balanced accuracy without clustering (LR): {balanced_accuracy_score(lr_noclust.predict(X_test), y_test)}")
print(f"Balanced accuracy with clustering (LR): {balanced_accuracy_score(lr_clust.predict(Xc_test), y_test)}")

Widzimy całkiem zauważalną poprawę balanced accuracy po dodaniu clusteryzacji

| Model              | BA Bez skupień | BA ze skupieniami |
|-------------------|----------------|--------------------|
| XGBClassifier      | 0.8845         | 0.8944             |
| LogisticRegression | 0.6834         | 0.6874             |

zbiór SPAM - klasyfikacja maili na zwykłe i spamowe - dużo danych

In [ ]:
dataset = openml.datasets.get_dataset(44)

X, y, _, _ = dataset.get_data(
    target=dataset.default_target_attribute
)

X_ptrain, X_test, y_ptrain, y_test = train_test_split(X, y, test_size=0.2, random_state=2137)
X_train, X_val, y_train, y_val = train_test_split(X_ptrain, y_ptrain, test_size=0.2, random_state=2137)

In [ ]:
params["n_cluster_lower"] = 2
params["n_cluster_upper"] = 30
cstudy = cluster_master.cluster_master_study(np.asarray(X_ptrain), "genie", "silhouette", 300, params)
clopt = genieclust.Genie(**cstudy.best_params)
clusters = clopt.fit_predict(X)
Xc = X.assign(cluster=clusters)
X_ptrain, X_test, Xc_ptrain, Xc_test, y_ptrain, y_test = train_test_split(X, Xc, y, test_size=0.2, random_state=2137)
X_train, X_val, Xc_train, Xc_val, y_train, y_val = train_test_split(X_ptrain, Xc_ptrain, y_ptrain, test_size=0.2, random_state=2137)

In [ ]:
y_ptrain, y_train, y_val, y_test = y_ptrain.astype(int), y_train.astype(int), y_val.astype(int), y_test.astype(int)

In [ ]:
study_xgb_noclust = optuna.create_study(direction="maximize")
study_xgb_noclust.optimize(lambda trial: objective_xgb(trial), n_trials=200)

xgb_nocl = XGBClassifier(
    **study_xgb_noclust.best_params
)
xgb_nocl.fit(X_ptrain, y_ptrain)

study_xgb_clust = optuna.create_study(direction="maximize")
study_xgb_clust.optimize(lambda trial: objective_xgb(trial, clust = "yes please"), n_trials=200)

xgb_clust = XGBClassifier(
    **study_xgb_clust.best_params
)
xgb_clust.fit(Xc_ptrain, y_ptrain)

In [ ]:
print(f"Balanced accuracy without clustering (XGB): {balanced_accuracy_score(xgb_nocl.predict(X_test), y_test)}")
print(f"Balanced accuracy with clustering (XGB): {balanced_accuracy_score(xgb_clust.predict(Xc_test), y_test)}")

lr_noclust = LogisticRegression(max_iter = 12000)
lr_noclust.fit(X_ptrain, y_ptrain)

lr_clust = LogisticRegression(max_iter = 12000)
lr_clust.fit(Xc_ptrain, y_ptrain)

print(f"Balanced accuracy without clustering (LR): {balanced_accuracy_score(lr_noclust.predict(X_test), y_test)}")
print(f"Balanced accuracy with clustering (LR): {balanced_accuracy_score(lr_clust.predict(Xc_test), y_test)}")

W tym przypadku skupienia nie poprawiły regresji logistycznej, a przy XGB poprawa jest niewielka. Duża liczba zmiennych objaśniających czyni skupienia "częściowo redundantnymi"

| Model              | BA Bez skupień | BA ze skupieniami |
|-------------------|----------------|--------------------|
| XGBClassifier      | 0.9485         | 0.9520             |
| LogisticRegression | 0.9252         | 0.9252             |

zbiór electricity - wzrosty/spadki cen prądu w Australii

In [ ]:
dataset = openml.datasets.get_dataset(151)

X, y, _, _ = dataset.get_data(
    target=dataset.default_target_attribute
)
y = (y=="UP").astype(int)
X_ptrain, X_test, y_ptrain, y_test = train_test_split(X, y, test_size=0.2, random_state=2137)
X_train, X_val, y_train, y_val = train_test_split(X_ptrain, y_ptrain, test_size=0.2, random_state=2137)

In [ ]:
params["n_cluster_lower"] = 2
params["n_cluster_upper"] = 30
params['min_samples_lower'] = 10
params['min_samples_upper'] = 100
params['eps_upper'] = 1.0

cstudy = cluster_master.cluster_master_study(np.asarray(X_ptrain).astype(float), "genie", "calinski-harabasz", 1000, params)
clopt = genieclust.Genie(**cstudy.best_params)
clusters = clopt.fit_predict(X)
Xc = X.assign(cluster=clusters)
X_ptrain, X_test, Xc_ptrain, Xc_test, y_ptrain, y_test = train_test_split(X, Xc, y, test_size=0.2, random_state=2137)
X_train, X_val, Xc_train, Xc_val, y_train, y_val = train_test_split(X_ptrain, Xc_ptrain, y_ptrain, test_size=0.2, random_state=2137)

In [ ]:
X_ptrain, X_train, X_val, X_test, Xc_ptrain, Xc_train, Xc_val, Xc_test = (np.asarray(X_ptrain).astype(float),
                                                                         np.asarray(X_train).astype(float),
                                                                         np.asarray(X_val).astype(float),
                                                                         np.asarray(X_test).astype(float),
                                                                         np.asarray(Xc_ptrain).astype(float),
                                                                         np.asarray(Xc_train).astype(float),
                                                                         np.asarray(Xc_val).astype(float),
                                                                         np.asarray(Xc_test).astype(float))

In [ ]:
study_xgb_noclust = optuna.create_study(direction="maximize")
study_xgb_noclust.optimize(lambda trial: objective_xgb(trial), n_trials=200)

xgb_nocl = XGBClassifier(
    **study_xgb_noclust.best_params
)
xgb_nocl.fit(X_ptrain, y_ptrain)

study_xgb_clust = optuna.create_study(direction="maximize")
study_xgb_clust.optimize(lambda trial: objective_xgb(trial, clust = "yes please"), n_trials=200)

xgb_clust = XGBClassifier(
    **study_xgb_clust.best_params
)
xgb_clust.fit(Xc_ptrain, y_ptrain)

In [ ]:
print(f"Balanced accuracy without clustering (XGB): {balanced_accuracy_score(xgb_nocl.predict(X_test), y_test)}")
print(f"Balanced accuracy with clustering (XGB): {balanced_accuracy_score(xgb_clust.predict(Xc_test), y_test)}")

lr_noclust = LogisticRegression(max_iter = 12000, l1_ratio=0.8, solver = "saga", penalty="elasticnet")
lr_noclust.fit(X_ptrain, y_ptrain)

lr_clust = LogisticRegression(max_iter = 12000, l1_ratio=0.8, solver = "saga", penalty="elasticnet")
lr_clust.fit(Xc_ptrain, y_ptrain)

print(f"Balanced accuracy without clustering (LR): {balanced_accuracy_score(lr_noclust.predict(X_test), y_test)}")
print(f"Balanced accuracy with clustering (LR): {balanced_accuracy_score(lr_clust.predict(Xc_test), y_test)}")

| Model              | BA Bez skupień | BA ze skupieniami |
|-------------------|----------------|--------------------|
| XGBClassifier      | 0.9374         | 0.9396             |
| LogisticRegression | 0.7592         | 0.7618             |

### Wnioski

1. Tworzenie skupień na ogół delikatnie pomaga w uczeniu nadzorowanym (algorytmy analizy skupień pozwalają znaleźć zależności, których metody uczenia nadzorowanego nie wykryły).

2. Poprawa jakości modelu jest raczej niewielka, acz zauważalna.

3. Skupienia są bardziej skuteczne, gdy mamy mało zmiennych objaśniających (przy dużej ich liczbie kolumna skupień może okazać się redundantna)/